# Practical 3: Time Series Analysis & Forecasting
## Dataset: Electric_Production.xls
## Objective: Perform multiplicative decomposition, Holt-Winters forecasting, and stationarity analysis on electricity production data

In [ ]:
import pandas as pd
import numpy as np 
import os
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
import seaborn as sns

In [ ]:
df = pd.read_csv('Electric_Production.xls')
df.head()

In [ ]:
df.shape

In [ ]:
# convert into date time
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.set_index('DATE')
df.head()

In [ ]:
sns.lineplot(df)
plt.ylabel('Electricity_Production')

### 📊 How to Read This Graph:
- Electricity production shows a **long-term upward trend** with occasional dips (economic cycles).
- There are clear **seasonal peaks** every winter (heating) and summer (cooling).
- The seasonal amplitude **expands** over time → **Multiplicative model** is appropriate.

In [ ]:
result = seasonal_decompose(df[['IPG2211A2N']],
                            model = 'multiplicative',
                            period = 12)
result.plot()
plt.show()

### 📊 How to Read This Decomposition:
- **Observed**: Raw electricity production index.
- **Trend**: Long-term economic growth in electricity demand.
- **Seasonal**: Annual 12-month cycle showing winter/summer demand peaks.
- **Resid**: Random unexplained fluctuations.

In [ ]:
import pymannkendall as mk
mk.original_test(df['IPG2211A2N'])

In [ ]:
#Train test splitting
train_df = df[:int(df.shape[0]*0.7)]
test_df = df[int(df.shape[0]*0.7):]

In [ ]:
from statsmodels.tsa.api import ExponentialSmoothing

model_triple_mul = ExponentialSmoothing(train_df['IPG2211A2N'], seasonal_periods = 12, trend = "add", seasonal = "mul")
model_triple_fit_mul = model_triple_mul.fit()

In [ ]:
forecast_triple_mul = model_triple_fit_mul.forecast(len(test_df))
print(forecast_triple_mul)

In [ ]:
plt.plot(df['IPG2211A2N'], label ="Original Data")
plt.plot(model_triple_fit_mul.fittedvalues, label= "Fitted values")
plt.plot(forecast_triple_mul, label = "Forecast")
plt.title("Holt-Winters Multiplicative - Electricity Production")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(test_df['IPG2211A2N'], forecast_triple_mul)
print("MAPE:", mape)

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

print("Raw ADF p-value:", adfuller(df['IPG2211A2N'])[1])
print("Raw KPSS p-value:", kpss(df['IPG2211A2N'])[1])

In [ ]:
#Combined Seasonal and Non-Seasonal Differencing
sddiff = df['IPG2211A2N'].diff(12).diff().dropna()

print("Differenced ADF p-value:", adfuller(sddiff)[1])
print("Differenced KPSS p-value:", kpss(sddiff)[1])